In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Silver 23: Staging Vessels (`project_staging.stg_vessels`)
# MAGIC
# MAGIC - Reads Bronze vessel JSON files (`batch_XXXX/part-*.json` and `*.jsonl`) using the verified all-string `StructType` schema + `_rescued_data`.
# MAGIC - Quarantines structurally empty/corrupt rows to `silver/_quarantine/schema_drift`.
# MAGIC - Explodes `selfReportedInfo` (with `registryInfo` fallback for registry-only vessels) and joins with the latest `combinedSourcesInfo` (`yearTo DESC`).
# MAGIC - Extracts scalar values from GFW metric arrays (`bestLengthM`, `bestTonnageGt`, `bestEnginePowerKw`, `maxSpeedKn`) and casts via Spark Connect-safe SQL `try_cast` (`DECIMAL(10,2)`).
# MAGIC - Normalizes vessel names (`InitCap`, cleans stray `>` / `&`, null -> `'Unknown'`) and uppercases ISO-3 flags / ship types / gear types.
# MAGIC - Routes records missing `vessel_id` to `silver/stg_rejects`.
# MAGIC - Deduplicates by `vessel_id` and executes an idempotent Delta `MERGE` into `silver/stg_vessels`.
# MAGIC - Extracts `numTargetRowsInserted` and `numTargetRowsUpdated` from Delta history and writes full run metrics to `silver/_audit/pipeline_run_log`.

# COMMAND ----------

from datetime import datetime, timezone
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    StringType,
    StructField,
    StructType,
)

# ── 1. Configuration ─────────────────────────────────────────────────────────
SA_NAME = "stmaritimeks"
BRONZE_ROOT = f"abfss://bronze@{SA_NAME}.dfs.core.windows.net"
SILVER_ROOT = f"abfss://silver@{SA_NAME}.dfs.core.windows.net"

BRONZE_VESSELS_PATH = f"{BRONZE_ROOT}/gfw/vessels/*/*/*/*.json*"
STG_VESSELS_PATH = f"{SILVER_ROOT}/stg_vessels"
STG_REJECTS_PATH = f"{SILVER_ROOT}/stg_rejects"
QUARANTINE_PATH = f"{SILVER_ROOT}/_quarantine/schema_drift"
AUDIT_LOG_PATH = f"{SILVER_ROOT}/_audit/pipeline_run_log"

# ── 2. Explicit Upfront All-String Bronze Read Schema (+ _rescued_data) ──────
BRONZE_VESSEL_ALL_STRING_SCHEMA = StructType([
    StructField("dataset", StringType(), True),
    StructField("registryInfoTotalRecords", StringType(), True),
    StructField("registryInfo", ArrayType(StructType([
        StructField("id", StringType(), True),
        StructField("sourceCode", ArrayType(StringType()), True),
        StructField("ssvid", StringType(), True),
        StructField("imo", StringType(), True),
        StructField("shipname", StringType(), True),
        StructField("nShipname", StringType(), True),
        StructField("callsign", StringType(), True),
        StructField("flag", StringType(), True),
        StructField("latestVesselInfo", StringType(), True),
        StructField("transmissionDateFrom", StringType(), True),
        StructField("transmissionDateTo", StringType(), True),
        StructField("geartypes", ArrayType(StringType()), True),
        StructField("lengthM", StringType(), True),
        StructField("tonnageGt", StringType(), True),
        StructField("enginePowerKw", StringType(), True),
        StructField("vesselInfoReference", StringType(), True),
        StructField("recordId", StringType(), True),
        StructField("extraFields", ArrayType(StructType([
            StructField("registrySource", StringType(), True),
            StructField("iuuStatus", StructType([
                StructField("value", StringType(), True),
                StructField("dateFrom", StringType(), True),
                StructField("dateFromMask", StringType(), True),
                StructField("dateTo", StringType(), True),
                StructField("dateToMask", StringType(), True),
            ]), True),
            StructField("hasComplianceInfo", StringType(), True),
            StructField("images", ArrayType(StringType()), True),
            StructField("operator", StringType(), True),
            StructField("builtYear", StringType(), True),
            StructField("depthM", StringType(), True),
        ])), True),
    ])), True),
    StructField("registryOwners", ArrayType(StructType([
        StructField("name", StringType(), True),
        StructField("flag", StringType(), True),
        StructField("ssvid", StringType(), True),
        StructField("sourceCode", ArrayType(StringType()), True),
        StructField("dateFrom", StringType(), True),
        StructField("dateTo", StringType(), True),
    ])), True),
    StructField("registryPublicAuthorizations", ArrayType(StructType([
        StructField("dateFrom", StringType(), True),
        StructField("dateTo", StringType(), True),
        StructField("sourceCode", ArrayType(StringType()), True),
        StructField("rfmo", StringType(), True),
        StructField("originalRfmo", StringType(), True),
    ])), True),
    StructField("selfReportedInfo", ArrayType(StructType([
        StructField("id", StringType(), True),
        StructField("ssvid", StringType(), True),
        StructField("shipname", StringType(), True),
        StructField("nShipname", StringType(), True),
        StructField("flag", StringType(), True),
        StructField("callsign", StringType(), True),
        StructField("imo", StringType(), True),
        StructField("transmissionDateFrom", StringType(), True),
        StructField("transmissionDateTo", StringType(), True),
        StructField("sourceCode", ArrayType(StringType()), True),
        StructField("matchFields", StringType(), True),
        StructField("index", StringType(), True),
        StructField("messagesCounter", StringType(), True),
        StructField("positionsCounter", StringType(), True),
        StructField("reference", StringType(), True),
    ])), True),
    StructField("combinedSourcesInfo", ArrayType(StructType([
        StructField("vesselId", StringType(), True),
        StructField("geartypes", ArrayType(StructType([
            StructField("name", StringType(), True),
            StructField("source", StringType(), True),
            StructField("yearFrom", StringType(), True),
            StructField("yearTo", StringType(), True),
        ])), True),
        StructField("shiptypes", ArrayType(StructType([
            StructField("name", StringType(), True),
            StructField("source", StringType(), True),
            StructField("yearFrom", StringType(), True),
            StructField("yearTo", StringType(), True),
        ])), True),
        StructField("bestLengthM", StringType(), True),
        StructField("bestTonnageGt", StringType(), True),
        StructField("bestEnginePowerKw", StringType(), True),
        StructField("maxSpeedKn", StringType(), True),
        StructField("yearFrom", StringType(), True),
        StructField("yearTo", StringType(), True),
    ])), True),
    StructField("_rescued_data", StringType(), True),
])

# COMMAND ----------

# ── 3. Unified Quarantine, Audit & Cleaning Helpers ──────────────────────────
def write_schema_drift_quarantine(df_quarantine, layer: str, entity: str, run_id: str) -> int:
    if df_quarantine.isEmpty():
        return 0
    count = df_quarantine.count()
    out = (
        df_quarantine
        .withColumn("quarantine_ts_utc", F.lit(datetime.now(timezone.utc).isoformat()))
        .withColumn("run_id", F.lit(run_id))
        .withColumn("layer", F.lit(layer))
        .withColumn("entity", F.lit(entity))
        .select(
            "quarantine_ts_utc",
            "run_id",
            "layer",
            "entity",
            F.col("_source_file").alias("source_file"),
            F.coalesce(F.col("_rescued_data"), F.lit("EMPTY_VESSEL_RECORD")).alias("rescued_payload"),
        )
    )
    out.write.format("delta").option("mergeSchema", "true").mode("append").save(QUARANTINE_PATH)
    return count


def log_pipeline_audit(
    run_id: str,
    layer: str,
    notebook_name: str,
    entity: str,
    status: str,
    started_at_utc: str,
    rows_in: int,
    rows_out: int,
    rows_inserted: int = 0,
    rows_updated: int = 0,
    quarantined_rows: int = 0,
    duplicates_removed: int = 0,
    error_message: str = None,
):
    audit_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("layer", StringType(), True),
        StructField("notebook_name", StringType(), True),
        StructField("entity", StringType(), True),
        StructField("status", StringType(), True),
        StructField("started_at_utc", StringType(), True),
        StructField("finished_at_utc", StringType(), True),
        StructField("rows_in", StringType(), True),
        StructField("rows_out", StringType(), True),
        StructField("rows_inserted", StringType(), True),
        StructField("rows_updated", StringType(), True),
        StructField("quarantined_rows", StringType(), True),
        StructField("duplicates_removed", StringType(), True),
        StructField("error_message", StringType(), True),
    ])
    row = [(
        run_id,
        layer,
        notebook_name,
        entity,
        status,
        started_at_utc,
        datetime.now(timezone.utc).isoformat(),
        str(rows_in),
        str(rows_out),
        str(rows_inserted),
        str(rows_updated),
        str(quarantined_rows),
        str(duplicates_removed),
        error_message,
    )]
    (
        spark.createDataFrame(row, schema=audit_schema)
        .write.format("delta")
        .option("mergeSchema", "true")
        .mode("append")
        .save(AUDIT_LOG_PATH)
    )


def clean_vessel_name(col_expr):
    cleaned = F.regexp_replace(F.trim(col_expr), r"[>&]+$", "")
    cleaned = F.regexp_replace(F.trim(cleaned), r"\s+", " ")
    return F.when(cleaned.isNull() | (cleaned == ""), F.lit("Unknown")).otherwise(F.initcap(cleaned))


def clean_upper_code(col_expr):
    cleaned = F.upper(F.trim(col_expr))
    return F.when(cleaned.isNull() | (cleaned == ""), F.lit(None).cast(StringType())).otherwise(cleaned)


def extract_best_metric(col_expr):
    """
    Extracts the numeric value string from either a scalar string ('64.2') or
    a GFW combinedSourcesInfo JSON array
    ('[{"value": 36.18, "source": "...", "yearFrom": 2024, "yearTo": 2024}]'),
    selecting the entry with the highest yearTo (Spark Connect compatible).
    """
    metric_arr_schema = ArrayType(StructType([
        StructField("value", StringType(), True),
        StructField("source", StringType(), True),
        StructField("yearFrom", StringType(), True),
        StructField("yearTo", StringType(), True),
    ]))
    parsed_arr = F.from_json(col_expr, metric_arr_schema)
    sorted_arr = F.array_sort(
        parsed_arr,
        lambda l, r: F.when(
            F.coalesce(F.regexp_extract(l["yearTo"], r"^(\d+)$", 1).cast("int"), F.lit(0))
            > F.coalesce(F.regexp_extract(r["yearTo"], r"^(\d+)$", 1).cast("int"), F.lit(0)),
            -1,
        ).when(
            F.coalesce(F.regexp_extract(l["yearTo"], r"^(\d+)$", 1).cast("int"), F.lit(0))
            < F.coalesce(F.regexp_extract(r["yearTo"], r"^(\d+)$", 1).cast("int"), F.lit(0)),
            1,
        ).otherwise(0),
    )
    return F.coalesce(F.get(sorted_arr, 0)["value"], col_expr)

# COMMAND ----------

# ── 4. Main Silver Transformation ────────────────────────────────────────────
def run_stg_vessels():
    started_at = datetime.now(timezone.utc).isoformat()
    run_id = f"silver_vessels_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    load_ts = F.current_timestamp()

    rows_in = 0
    rows_out = 0
    rows_inserted = 0
    rows_updated = 0
    quarantined_rows = 0
    duplicates_removed = 0

    try:
        # 1. Read Bronze with explicit all-string schema + _rescued_data
        df_raw = (
            spark.read
            .schema(BRONZE_VESSEL_ALL_STRING_SCHEMA)
            .option("mode", "PERMISSIVE")
            .option("primitivesAsString", "true")
            .option("rescuedDataColumn", "_rescued_data")
            .json(BRONZE_VESSELS_PATH)
            .withColumn("_source_file", F.col("_metadata.file_path"))
            .withColumn("_bronze_ingest_ts", F.col("_metadata.file_modification_time"))
        )
        rows_in = df_raw.count()

        # 2. Quarantine structurally empty/corrupt rows to schema_drift table
        has_vessel_data = (
            (F.size(F.coalesce(F.col("selfReportedInfo"), F.array())) > 0)
            | (F.size(F.coalesce(F.col("registryInfo"), F.array())) > 0)
        )
        df_broken = df_raw.filter(~has_vessel_data)
        quarantined_rows = write_schema_drift_quarantine(df_broken, "silver", "stg_vessels", run_id)

        # Keep clean structurally valid rows (ANSI-safe F.get instead of [0])
        df_clean_raw = df_raw.filter(has_vessel_data).withColumn(
            "fallback_vessel_id",
            F.coalesce(
                F.get(F.col("selfReportedInfo"), 0)["id"],
                F.get(F.col("registryInfo"), 0)["id"],
            ),
        )

        # 3. Extract latest combinedSourcesInfo per vessel (ordered by yearTo DESC)
        df_combined = (
            df_clean_raw
            .select(
                F.col("fallback_vessel_id"),
                F.explode_outer("combinedSourcesInfo").alias("csi"),
            )
            .withColumn("_csi_year_raw", F.col("csi.yearTo"))
            .withColumn("csi_year_to", F.coalesce(F.expr("try_cast(_csi_year_raw AS INT)"), F.lit(0)))
            .withColumn(
                "rn",
                F.row_number().over(
                    Window.partitionBy(F.coalesce(F.col("csi.vesselId"), F.col("fallback_vessel_id")))
                    .orderBy(F.col("csi_year_to").desc())
                ),
            )
            .filter(F.col("rn") == 1)
            .select(
                F.coalesce(F.col("csi.vesselId"), F.col("fallback_vessel_id")).alias("csi_vessel_id"),
                F.col("csi.bestLengthM").alias("raw_length_m"),
                F.col("csi.bestTonnageGt").alias("raw_tonnage_gt"),
                F.col("csi.bestEnginePowerKw").alias("raw_engine_power_kw"),
                F.col("csi.maxSpeedKn").alias("raw_max_speed_kn"),
                F.transform(F.col("csi.shiptypes"), lambda x: F.upper(F.trim(x["name"]))).alias("ship_types"),
                F.transform(F.col("csi.geartypes"), lambda x: F.upper(F.trim(x["name"]))).alias("gear_types"),
            )
        )

        # 4. Explode selfReportedInfo (with registryInfo[0] fallback when selfReportedInfo is empty)
        df_sri = (
            df_clean_raw
            .select(
                F.col("fallback_vessel_id"),
                F.get(F.col("registryInfo"), 0).alias("reg0"),
                F.explode_outer("selfReportedInfo").alias("sri"),
                "_source_file",
                "_bronze_ingest_ts",
            )
            .select(
                F.coalesce(F.col("sri.id"), F.col("reg0.id"), F.col("fallback_vessel_id")).alias("vessel_id"),
                F.coalesce(F.col("sri.shipname"), F.col("reg0.shipname")).alias("raw_shipname"),
                F.coalesce(F.col("sri.ssvid"), F.col("reg0.ssvid")).alias("raw_ssvid"),
                F.coalesce(F.col("sri.imo"), F.col("reg0.imo")).alias("raw_imo"),
                F.coalesce(F.col("sri.callsign"), F.col("reg0.callsign")).alias("raw_callsign"),
                F.coalesce(F.col("sri.flag"), F.col("reg0.flag")).alias("raw_flag"),
                F.coalesce(F.col("sri.transmissionDateTo"), F.col("reg0.transmissionDateTo")).alias("_raw_tx_to"),
                F.col("reg0.lengthM").alias("reg_length_m"),
                F.col("reg0.tonnageGt").alias("reg_tonnage_gt"),
                F.col("reg0.enginePowerKw").alias("reg_engine_power_kw"),
                F.transform(F.col("reg0.geartypes"), lambda g: F.upper(F.trim(g))).alias("reg_gear_types"),
                "_source_file",
                "_bronze_ingest_ts",
            )
            .withColumn("transmission_to_ts", F.expr("try_cast(_raw_tx_to AS TIMESTAMP)"))
            .drop("_raw_tx_to")
        )

        # 5. Join SRI with latest CSI, extract array metrics, and ANSI-safe SQL try_cast
        df_joined = (
            df_sri
            .join(df_combined, df_sri["vessel_id"] == df_combined["csi_vessel_id"], how="left")
            .select(
                F.trim(F.col("vessel_id")).alias("vessel_id"),
                clean_vessel_name(F.col("raw_shipname")).alias("vessel_name"),
                clean_upper_code(F.col("raw_ssvid")).alias("ssvid"),
                clean_upper_code(F.col("raw_imo")).alias("imo"),
                clean_upper_code(F.col("raw_callsign")).alias("callsign"),
                clean_upper_code(F.col("raw_flag")).alias("vessel_flag"),
                F.coalesce(extract_best_metric(F.col("raw_length_m")), F.col("reg_length_m")).alias("_len_str"),
                F.coalesce(extract_best_metric(F.col("raw_tonnage_gt")), F.col("reg_tonnage_gt")).alias("_ton_str"),
                F.coalesce(extract_best_metric(F.col("raw_engine_power_kw")), F.col("reg_engine_power_kw")).alias("_pwr_str"),
                extract_best_metric(F.col("raw_max_speed_kn")).alias("_spd_str"),
                F.coalesce(F.col("ship_types"), F.array().cast(ArrayType(StringType()))).alias("ship_types"),
                F.coalesce(F.col("gear_types"), F.col("reg_gear_types"), F.array().cast(ArrayType(StringType()))).alias("gear_types"),
                F.col("_source_file"),
                F.col("_bronze_ingest_ts"),
                load_ts.alias("_silver_load_ts"),
                F.lit(run_id).alias("_run_id"),
                F.col("transmission_to_ts"),
            )
            .withColumn("length_m", F.expr("try_cast(_len_str AS DECIMAL(10, 2))"))
            .withColumn("tonnage_gt", F.expr("try_cast(_ton_str AS DECIMAL(10, 2))"))
            .withColumn("engine_power_kw", F.expr("try_cast(_pwr_str AS DECIMAL(10, 2))"))
            .withColumn("max_speed_kn", F.expr("try_cast(_spd_str AS DECIMAL(10, 2))"))
            .drop("_len_str", "_ton_str", "_pwr_str", "_spd_str")
            .select(
                "vessel_id",
                "vessel_name",
                "ssvid",
                "imo",
                "callsign",
                "vessel_flag",
                "length_m",
                "tonnage_gt",
                "engine_power_kw",
                "max_speed_kn",
                "ship_types",
                "gear_types",
                "_source_file",
                "_bronze_ingest_ts",
                "_silver_load_ts",
                "_run_id",
                "transmission_to_ts",
            )
        )

        # 6. Split critical rejects (missing vessel_id) -> stg_rejects
        df_bad = df_joined.filter(F.col("vessel_id").isNull() | (F.col("vessel_id") == ""))
        if not df_bad.isEmpty():
            df_reject_out = df_bad.select(
                F.expr("uuid()").alias("reject_id"),
                F.lit("stg_vessels").alias("target_table"),
                F.lit("MISSING_VESSEL_ID").alias("reject_reason"),
                F.to_json(F.struct("*")).alias("raw_payload"),
                F.col("_source_file"),
                load_ts.alias("_rejected_at_utc"),
                F.lit(run_id).alias("_run_id"),
            )
            df_reject_out.write.format("delta").option("mergeSchema", "true").mode("append").save(STG_REJECTS_PATH)

        df_valid_pre_dedup = df_joined.filter(F.col("vessel_id").isNotNull() & (F.col("vessel_id") != ""))
        pre_dedup_count = df_valid_pre_dedup.count()

        # 7. Deduplicate by vessel_id keeping latest _bronze_ingest_ts & transmission_to_ts
        dedup_win = Window.partitionBy("vessel_id").orderBy(
            F.col("_bronze_ingest_ts").desc_nulls_last(),
            F.col("transmission_to_ts").desc_nulls_last(),
        )
        df_silver = (
            df_valid_pre_dedup
            .withColumn("_rn", F.row_number().over(dedup_win))
            .filter(F.col("_rn") == 1)
            .drop("_rn", "transmission_to_ts")
        )
        rows_out = df_silver.count()
        duplicates_removed = pre_dedup_count - rows_out

        # 8. Idempotent Delta MERGE into silver/stg_vessels & extract operationMetrics
        spark.sql("CREATE SCHEMA IF NOT EXISTS project_staging")
        if DeltaTable.isDeltaTable(spark, STG_VESSELS_PATH):
            target_dt = DeltaTable.forPath(spark, STG_VESSELS_PATH)
            (
                target_dt.alias("t")
                .merge(df_silver.alias("s"), "t.vessel_id = s.vessel_id")
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
            metrics = (
                target_dt.history(1)
                .select("operationMetrics")
                .first()["operationMetrics"]
            )
            rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
            rows_updated = int(metrics.get("numTargetRowsUpdated", 0))
        else:
            df_silver.write.format("delta").mode("overwrite").save(STG_VESSELS_PATH)
            rows_inserted = rows_out
            rows_updated = 0

        try:
            spark.sql("CREATE SCHEMA IF NOT EXISTS project_staging")
            spark.sql(f"""
                CREATE TABLE IF NOT EXISTS project_staging.stg_vessels
                USING DELTA
                LOCATION '{STG_VESSELS_PATH}'
            """)
        except Exception as uc_err:
            print(f"[WARN] Delta data written to {STG_VESSELS_PATH}, awaiting Member 1 UC GRANT: {uc_err}")

        log_pipeline_audit(
            run_id=run_id,
            layer="silver",
            notebook_name="23_stg_vessels",
            entity="stg_vessels",
            status="SUCCESS",
            started_at_utc=started_at,
            rows_in=rows_in,
            rows_out=rows_out,
            rows_inserted=rows_inserted,
            rows_updated=rows_updated,
            quarantined_rows=quarantined_rows,
            duplicates_removed=duplicates_removed,
        )
        print(
            f"[SUCCESS] stg_vessels merged: rows_in={rows_in}, rows_out={rows_out}, "
            f"inserted={rows_inserted}, updated={rows_updated}, "
            f"duplicates_removed={duplicates_removed}, quarantined={quarantined_rows}"
        )

    except Exception as e:
        log_pipeline_audit(
            run_id=run_id,
            layer="silver",
            notebook_name="23_stg_vessels",
            entity="stg_vessels",
            status="FAILED",
            started_at_utc=started_at,
            rows_in=rows_in,
            rows_out=rows_out,
            rows_inserted=rows_inserted,
            rows_updated=rows_updated,
            quarantined_rows=quarantined_rows,
            duplicates_removed=duplicates_removed,
            error_message=str(e)[:500],
        )
        raise


if __name__ == "__main__":
    run_stg_vessels()